# Fine-tune Gemma 4 E2B on HealthCareMagic-100k

For the healthOS on-device chat model. Run top-to-bottom on Kaggle Notebooks with **GPU T4 x2** accelerator.

**Before running:**
1. Accept the Gemma 4 license in a browser: <https://huggingface.co/google/gemma-4-E2B-it> → "Agree and access repository".
2. Attach your `HF_TOKEN` (write scope) via Add-ons → Secrets.
3. In the CONFIG cell below, set your Hugging Face username.

**Wall time:** ~4-5 h end-to-end on 2× T4 (mostly training + upload).

## 1. Config — set your HF username here

In [ ]:
# EDIT THESE TWO LINES
HF_USERNAME = "your-hf-username"           # <-- change to your Hugging Face username
TUNED_MODEL_NAME = "gemma-4-E2B-healthcare"  # <-- ok to leave as-is

# Derived (do not edit)
BASE_MODEL = "google/gemma-4-E2B-it"
MERGED_REPO_ID = f"{HF_USERNAME}/{TUNED_MODEL_NAME}"
MAX_SEQ_LEN = 2048
TRAIN_ROWS = 3000
LORA_RANK = 16

print(f"Will fine-tune  : {BASE_MODEL}")
print(f"Will save to    : {MERGED_REPO_ID}")
print(f"Training rows   : {TRAIN_ROWS}")
assert HF_USERNAME != "your-hf-username", "Set your HF_USERNAME above before running the rest."

## 2. Install dependencies

Takes ~3-5 minutes on Kaggle. Unsloth gives us the fastest QLoRA path on Gemma 4.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install --no-deps xformers==0.0.28.post2
!pip install --upgrade datasets huggingface_hub

## 3. Log in to Hugging Face

Pulls the `HF_TOKEN` secret you attached to this notebook. If this fails, add the secret via Add-ons → Secrets → `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("Hugging Face login OK.")

## 4. Load base Gemma 4 E2B with QLoRA

First download of the model is ~5 GB — takes 5-10 minutes on Kaggle.

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto (bf16 where supported, else fp16)
    load_in_4bit=True,   # QLoRA base
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(model.print_trainable_parameters())

## 5. Load + format HealthCareMagic-100k dataset

3,000 patient-doctor Q&A pairs, formatted with Gemma's chat template so tokens match what `buildContext.ts` produces at inference time.

Dataset: `lavita/ChatDoctor-HealthCareMagic-100k` (CC BY-SA 4.0).

In [ ]:
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
ds = ds.shuffle(seed=42).select(range(TRAIN_ROWS))

def format_example(row):
    prompt = row.get("instruction") or row.get("input") or ""
    answer = row.get("output") or row.get("response") or ""
    return {
        "text": (
            f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
            f"<start_of_turn>model\n{answer}<end_of_turn>"
        )
    }

ds = ds.map(format_example, remove_columns=ds.column_names)
print(f"Dataset ready. {len(ds)} rows.")
print("---Sample row---")
print(ds[0]["text"][:500])

## 6. Train — 1 epoch QLoRA

**This is the long step: ~2-3 hours on 2× T4.** You can close the browser tab and come back later.

Watch the loss column — expect start ~2.5-3.0, converge to ~1.4-1.7. If loss plateaus above 2.0, kill it and drop the learning rate to 1e-4.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/outputs",
        save_strategy="epoch",
        report_to="none",
    ),
)

stats = trainer.train()
print(stats)

## 7. Save LoRA, merge, push to Hugging Face

Upload is ~15-30 minutes for the ~5 GB merged checkpoint. Progress will show in the cell output.

In [ ]:
LORA_DIR = "/kaggle/working/gemma-4-e2b-healthcare-lora"
MERGED_DIR = "/kaggle/working/gemma-4-e2b-healthcare-merged"

# Save adapter locally
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("LoRA adapter saved.")

# Merge to full-weights checkpoint
print("Merging LoRA into base weights...")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("Merged checkpoint saved.")

# Push merged model to HF
from huggingface_hub import HfApi, create_repo

create_repo(MERGED_REPO_ID, repo_type="model", exist_ok=True)
HfApi().upload_folder(
    folder_path=MERGED_DIR,
    repo_id=MERGED_REPO_ID,
    repo_type="model",
)
print(f"Pushed to https://huggingface.co/{MERGED_REPO_ID}")

## 8. Vibe check — base vs tuned on 10 clinical prompts

Reads out both models' answers side-by-side. If the tuned answers look more clinical and less hedgy, you have a keeper. If they're worse, **stop here** — do not spend time on LiteRT conversion below.

In [ ]:
PROMPTS = [
    "I've been coughing for 5 days with yellow phlegm. Should I see a doctor?",
    "What are common side effects of metformin 500mg twice daily?",
    "My HbA1c went from 7.0 to 7.4 in a year — is that concerning?",
    "Can I take ibuprofen while on losartan for blood pressure?",
    "I feel dizzy every morning when I stand up. What could this be?",
    "My father has type 2 diabetes. Am I at higher risk?",
    "How often should I get a lipid panel done if I'm on atorvastatin?",
    "What does an LDL of 122 mean for someone with diabetes?",
    "I missed two days of my blood pressure medication. Should I double up today?",
    "My resting heart rate has been 78-82 bpm for a week. Is this abnormal?",
]

FastLanguageModel.for_inference(model)

def ask(prompt: str) -> str:
    text = (
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outs = model.generate(**inputs, max_new_tokens=200, temperature=0.6, do_sample=True)
    reply = tokenizer.decode(outs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return reply.strip()

print("=== TUNED MODEL RESPONSES ===\n")
for i, p in enumerate(PROMPTS, 1):
    print(f"[Q{i}] {p}")
    print(f"[A{i}] {ask(p)}")
    print("-" * 60)

## 9. Attempt LiteRT conversion — `.litertlm` for on-device

**This is the risky step.** Google's `ai-edge-torch` LLM converter is preview-tier. If this fails, the merged HF checkpoint from Step 7 is still usable via Vertex AI or vLLM — see the training/README.md fallback path.

Do not spend more than 90 minutes here. If the cell errors out with something like "unsupported model type" or "conversion failed", stop and use the fallback.

In [ ]:
%%capture
!pip install ai-edge-torch ai-edge-litert

In [ ]:
# ai-edge-torch exposes a Python API for LLM conversion. Exact API surface
# may drift — check https://ai.google.dev/edge/litert for the current version.

OUTPUT_LITERTLM = f"/kaggle/working/{TUNED_MODEL_NAME}.litertlm"

try:
    import ai_edge_torch
    from ai_edge_torch.generative.utilities import converter

    converter.convert_to_tflite(
        model_path=MERGED_DIR,
        output_path=OUTPUT_LITERTLM,
        quantize="dynamic_int4",
    )
    print(f"LiteRT file written to {OUTPUT_LITERTLM}")

    import os
    size_gb = os.path.getsize(OUTPUT_LITERTLM) / 1e9
    print(f"Size: {size_gb:.2f} GB")

    CONVERSION_OK = True
except Exception as e:
    print("LiteRT conversion FAILED:")
    print(repr(e))
    print("\nFall back to Vertex/vLLM deployment — see training/README.md step 6, Fallback A.")
    print("Your merged HF checkpoint at:", MERGED_REPO_ID, "is fully usable for cloud deployment.")
    CONVERSION_OK = False

## 10. Push `.litertlm` to Hugging Face (only if conversion succeeded)

In [ ]:
if CONVERSION_OK:
    HfApi().upload_file(
        path_or_fileobj=OUTPUT_LITERTLM,
        path_in_repo=f"{TUNED_MODEL_NAME}.litertlm",
        repo_id=MERGED_REPO_ID,
        repo_type="model",
    )
    print(f"Uploaded to https://huggingface.co/{MERGED_REPO_ID}/blob/main/{TUNED_MODEL_NAME}.litertlm")
    print()
    print("Copy this into your app's modelRegistry.ts:")
    print(f'  downloadUrl: "https://huggingface.co/{MERGED_REPO_ID}/resolve/main/{TUNED_MODEL_NAME}.litertlm"')
else:
    print("Skipped — conversion did not succeed. Use the merged HF checkpoint via cloud instead.")

## Done

If the `.litertlm` was produced, follow **step 7 in `training/README.md`** to wire it into the app. If it wasn't, follow the fallback path in **step 6**.

Don't forget to attribute the dataset in your pitch and repo docs — HealthCareMagic-100k is CC BY-SA 4.0.